# DL_01: Data Preparation & Preprocessing Pipeline
**Causal NLP Reasoning — Phase 2: Deep Learning**

---

## 1. Purpose and Scope
This notebook establishes reproducible data preparation and preprocessing pipelines for both Phase 2 benchmark streams:
1. **Corr2Cause**: Abstract causal inference formulated as Natural Language Inference (NLI) premise-hypothesis entailment pairs.
2. **BioCause**: Candidate-conditioned directed causal relation classification using annotated argument spans from open-access PMC biomedical articles.

Key requirements:
- Prepare Corr2Cause and BioCause datasets independently (strict boundary enforcement).
- Document dataset schemas, label distributions, and directional label semantics.
- Ingest canonical input files from `data/raw/` and `data/processed/`.
- Preserve the canonical Corr2Cause split counts: Train (205,734), Validation (1,076), and Test (1,162).
- Maintain the 12-train / 4-validation / 4-test PMC article-level split isolation for BioCause to prevent cross-article data leakage.
- Construct explicit argument boundary markers (`[E1]...[/E1]` and `[E2]...[/E2]`) preserving original argument order and contextual text.
- Define expected model input tensors and target outputs for transformer fine-tuning in `DL_02` and `DL_03`.


## 2. Experimental Methodology & Theoretical Background

### Causal Relation Classification vs. Causal Entailment
- **Corr2Cause (Symbolic NLI)**: Evaluates whether language models deduce valid causal graph relations solely from statistical independence statements:
  - **Premise**: Verbalized statistical independence constraints among variables.
  - **Hypothesis**: Directed causal conjecture (e.g. "*A causes B.*")
  - **Target Label**: `1` (Entailed) or `0` (Refuted).
  - **Canonical Split Distribution**: Train = 205,734; Validation = 1,076; Test = 1,162. Validation must remain strictly separate from training.

- **BioCause (Biomedical Candidate-Conditioned Relation Classification)**: Given co-occurring biomedical entity spans ($Arg_1$, $Arg_2$) in scientific text, classify whether the directed relationship $Arg_1 \to Arg_2$ holds:
  - **Label Semantics**: Label `1` denotes that $Arg_1$ causes $Arg_2$. Label `0` denotes that $Arg_1$ does not represent the gold directed relation $Arg_1 \to Arg_2$ under the candidate-generation convention (which includes reversed directional pairs $(Arg_2, Arg_1)$ as well as non-causal pairs). Label `0` must not be mischaracterized as universally non-causal.
  - **Class Imbalance**: Raw candidate pairing produces 13,408 candidate pairs (10,086 train, 1,914 validation, 1,408 test). After excluding 202 candidates with invalid source reconstructions (158 train, 30 validation, 14 test), **13,206 model-eligible candidates** remain: 9,928 train (621 positive, 9,307 negative), 1,884 validation (131 positive, 1,753 negative), and 1,394 test (100 positive, 1,294 negative). The training positive class weight computed strictly on model-eligible training data is $9,307 / 621 = 14.9871$.

### Article-Level Split Isolation (Zero Data Leakage)
BioCause strictly partitions the 20 PMC publications by article ID:
- **Train Split**: 12 PMC Articles — 10,086 raw candidates; **9,928 model-eligible** (621 positive, 9,307 negative; positive class weight $w_{\text{pos}} = 9,307 / 621 = 14.9871$)
- **Validation Split**: 4 PMC Articles — 1,914 raw candidates; **1,884 model-eligible** (131 positive, 1,753 negative)
- **Test Split**: 4 PMC Articles — 1,408 raw candidates; **1,394 model-eligible** (100 positive, 1,294 negative)


## 3. Expected Inputs

| Stream | File Path | Canonical Size / Dimensions | Key Fields |
| :--- | :--- | :--- | :--- |
| **Corr2Cause Train** | `data/raw/train.csv` | 205,734 rows | `input, label, num_variables, template` |
| **Corr2Cause Val** | `data/raw/validation.csv` | 1,076 rows | `input, label, num_variables, template` |
| **Corr2Cause Test** | `data/raw/test.csv` | 1,162 rows | `input, label, num_variables, template` |
| **BioCause Candidates** | `data/processed/biocause_candidates.csv` | 13,408 raw candidate pairs (13,206 model-eligible after source validation) | `candidate_id, article_id, section_id, arg1_text, arg2_text, between_text, label, split...` |
| **BioCause Gold Relations** | `data/processed/biocause_relations.csv` | 852 gold relations | `relation_id, article_id, cause_id, effect_id...` |


## 4. Technical Responsibilities & Implementation Requirements

1. **Environment & Portable Root Discovery**: Locate repo root dynamically and load `dl_utils`.
2. **Corr2Cause Split Ingestion**: Verify exact row counts (205,734 train, 1,076 val, 1,162 test) and verify binary label balance.
3. **BioCause Article-Level Isolation Audit**: Check that article IDs across splits are strictly disjoint.
4. **Argument Marker Construction**: Format BioCause inputs with special tokens `[E1]`, `[/E1]`, `[E2]`, `[/E2]`. Ensure markers reference original text offsets, preserve original text order, include surrounding context, and check for span validity/overlap.
5. **Dynamic Class Weight Calculation**: Compute $w_{\text{pos}} = N_{\text{neg}} / N_{\text{pos}}$ strictly on the model-eligible training split ($9,307 / 621 = 14.9871$). Never leak validation or test counts into weight computation.
6. **Token Length Profiling**: Profile sequence lengths under `distilbert-base-uncased` and `microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext` tokenizers.
7. **Dataset Serialization**: Export clean model-ready splits to `data/processed/`.


---

## Section A — Environment Setup
Set up repository imports, configure reproducibility, and detect compute device.

In [1]:
# Section A: Environment Setup & Portable Imports
import sys
import os
import warnings
from pathlib import Path

# ---- Repository root discovery ----
def find_repo_root():
    """Locate repository root across local, Colab, and Kaggle."""
    curr = Path(os.getcwd()).resolve()
    preferred_roots = [Path('/content/causal-nlp-reasoning'), Path('/workspace/causal-nlp-reasoning'), Path('/kaggle/working/causal-nlp-reasoning')]
    search_roots = [*preferred_roots, curr, Path('/content'), Path('/workspace'), Path('/kaggle/working')]
    checked = set()
    for base in search_roots:
        if not base.exists():
            continue
        candidates = [base, *base.parents]
        if base in {Path('/content'), Path('/workspace'), Path('/kaggle/working')}:
            candidates.extend(path.parent.parent for path in base.rglob('DL/dl_utils.py'))
        for candidate in candidates:
            candidate = candidate.resolve()
            if candidate in checked:
                continue
            checked.add(candidate)
            if ((candidate / 'AML' / 'causal_extractors.py').is_file()
                    and (candidate / 'DL' / 'dl_utils.py').is_file()):
                return candidate
    raise FileNotFoundError(
        'Repository root not found. Make the project folder available in the Colab workspace.'
    )

ROOT_DIR = find_repo_root()
for p in [ROOT_DIR, ROOT_DIR / 'AML', ROOT_DIR / 'DL']:
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

import dl_utils
import pandas as pd
import numpy as np



In [2]:
# Reproducibility & device detection
dl_utils.set_seed(42)
DEVICE = dl_utils.get_device()

---

## Section B — Corr2Cause Data Preparation
Load canonical Corr2Cause CSV splits and verify row counts, columns, labels, missing values, and duplicates.

In [3]:
# Section B: Corr2Cause Data Ingestion & Split Verification

CORR2CAUSE_RAW_DIR = ROOT_DIR / 'data' / 'raw'

c2c_files = {
    'train': CORR2CAUSE_RAW_DIR / 'train.csv',
    'validation': CORR2CAUSE_RAW_DIR / 'validation.csv',
    'test': CORR2CAUSE_RAW_DIR / 'test.csv',
}

# Check that all files exist before proceeding
missing_c2c = [name for name, path in c2c_files.items() if not path.exists()]
if missing_c2c:
    msg = (
        f'MISSING Corr2Cause raw files: {missing_c2c}\n'
        f'Expected location: data/raw/\n'
        f'Required files: train.csv, validation.csv, test.csv\n'
        f'Download from HuggingFace: https://huggingface.co/datasets/causalnlp/corr2cause\n'
        f'Or run 00_setup.ipynb to automatically stage the datasets.'
    )
    raise FileNotFoundError(msg)

In [4]:
# Load Corr2Cause splits
df_c2c_train = pd.read_csv(c2c_files['train'])
df_c2c_val = pd.read_csv(c2c_files['validation'])
df_c2c_test = pd.read_csv(c2c_files['test'])

# Verify exact row counts
assert len(df_c2c_train) == dl_utils.CORR2CAUSE_SPLIT_COUNTS['train'], \
    f'Train count mismatch: {len(df_c2c_train)} vs {dl_utils.CORR2CAUSE_SPLIT_COUNTS["train"]}'
assert len(df_c2c_val) == dl_utils.CORR2CAUSE_SPLIT_COUNTS['validation'], \
    f'Val count mismatch: {len(df_c2c_val)} vs {dl_utils.CORR2CAUSE_SPLIT_COUNTS["validation"]}'
assert len(df_c2c_test) == dl_utils.CORR2CAUSE_SPLIT_COUNTS['test'], \
    f'Test count mismatch: {len(df_c2c_test)} vs {dl_utils.CORR2CAUSE_SPLIT_COUNTS["test"]}'

In [5]:
# Validate required columns, binary labels, and absence of nulls
C2C_REQUIRED_COLS = ['input', 'label']
for split_name, df in [('train', df_c2c_train), ('val', df_c2c_val), ('test', df_c2c_test)]:
    dl_utils.validate_dataset_schema(df, C2C_REQUIRED_COLS, f'Corr2Cause-{split_name}')
    unique_labels = set(df['label'].dropna().unique())
    assert unique_labels.issubset({0, 1}), f'{split_name}: non-binary labels found: {unique_labels}'
    null_input = df['input'].isna().sum()
    null_label = df['label'].isna().sum()
    assert null_input == 0, f'{split_name}: {null_input} null inputs'
    assert null_label == 0, f'{split_name}: {null_label} null labels'

---

## Section C — BioCause Data Preparation
Load BioCause candidate pairs, verify article-level split isolation, candidate identifiers,
argument text/offsets, binary labels, and expected counts.

In [6]:
# Section C: BioCause Data Ingestion & Article-Level Split Isolation Audit

BIOCAUSE_CANDS_PATH = ROOT_DIR / 'data' / 'processed' / 'biocause_candidates.csv'

if not BIOCAUSE_CANDS_PATH.exists():
    msg = (
        f'MISSING BioCause candidates file: {BIOCAUSE_CANDS_PATH}\n'
        f'This file is generated by 00_setup.ipynb and AML/AML_01_data_features.ipynb.\n'
        f'Run those notebooks first to produce the candidate pairs.'
    )
    raise FileNotFoundError(msg)

df_bio = pd.read_csv(BIOCAUSE_CANDS_PATH, keep_default_na=False)

In [7]:
# Validate BioCause candidates
bio_summary = dl_utils.validate_biocause_candidates(df_bio, 'BioCause Candidates')

In [8]:
# Verify article-level split isolation explicitly
train_articles = set(df_bio[df_bio['split'] == 'train']['article_id'].unique())
val_articles = set(df_bio[df_bio['split'] == 'validation']['article_id'].unique())
test_articles = set(df_bio[df_bio['split'] == 'test']['article_id'].unique())

assert len(train_articles) == dl_utils.BIOCAUSE_ARTICLE_SPLITS['train'], (
    f'Expected 12 train articles, got {len(train_articles)}'
)
assert len(val_articles) == dl_utils.BIOCAUSE_ARTICLE_SPLITS['validation'], (
    f'Expected 4 validation articles, got {len(val_articles)}'
)
assert len(test_articles) == dl_utils.BIOCAUSE_ARTICLE_SPLITS['test'], (
    f'Expected 4 test articles, got {len(test_articles)}'
)
assert len(train_articles & val_articles) == 0, 'Train/Val article overlap detected!'
assert len(train_articles & test_articles) == 0, 'Train/Test article overlap detected!'
assert len(val_articles & test_articles) == 0, 'Val/Test article overlap detected!'

In [9]:
# Verify expected raw split counts against canonical values
expected = dl_utils.BIOCAUSE_CANDIDATE_COUNTS

for split_name in ['train', 'validation', 'test']:
    actual = bio_summary[split_name]
    exp = expected[split_name]
    assert actual['total'] == exp['total'], \
        f'{split_name} total: {actual["total"]} vs expected {exp["total"]}'
    assert actual['positive'] == exp['positive'], \
        f'{split_name} positive: {actual["positive"]} vs expected {exp["positive"]}'
    assert actual['negative'] == exp['negative'], \
        f'{split_name} negative: {actual["negative"]} vs expected {exp["negative"]}'

In [10]:
# Check for duplicate candidate IDs and verify argument identifiers/text are non-null
dup_count = df_bio['candidate_id'].duplicated().sum()
assert dup_count == 0, f'Found {dup_count} duplicate candidate_ids'

for col in ['arg1_id', 'arg2_id', 'arg1_text', 'arg2_text']:
    null_count = df_bio[col].isna().sum()
    assert null_count == 0, f'{col} has {null_count} null values'

---

## Section D — Entity Marker Construction
Insert `[E1]...[/E1]` and `[E2]...[/E2]` entity boundary markers using character offsets.
Verify that Arg1 and Arg2 identities are maintained and demonstrate handling of different argument orders.

In [11]:
# Section D: Entity Marker Construction via Source Section Reconstruction
# Check required candidate columns
required_cand_cols = ["arg1_start", "arg1_end", "arg2_start", "arg2_end", "arg1_text", "arg2_text", "section_id"]
missing_cand_cols = [c for c in required_cand_cols if c not in df_bio.columns]
if missing_cand_cols:
    raise ValueError(f"BioCause candidates missing required offset/span columns: {missing_cand_cols}")

# Locate and cache raw section .txt files from data/raw/biocause/BioCause_corpus/ (or fallback data/raw/biocause/)
unique_sec_ids = df_bio["section_id"].unique()

sec_text_cache = {}
missing_sec_files = []

for sec_id in unique_sec_ids:
    try:
        sec_text = dl_utils.load_biocause_section_text(
            sec_id,
            raw_dir=ROOT_DIR / 'data' / 'raw' / 'biocause',
        )
        sec_text_cache[sec_id] = sec_text
    except FileNotFoundError:
        missing_sec_files.append(sec_id)

if missing_sec_files:
    raise FileNotFoundError(
        f"Missing {len(missing_sec_files)} BioCause raw section .txt files in "
        f"data/raw/biocause/BioCause_corpus/ or data/raw/biocause/. "
        f"Sample missing sections: {missing_sec_files[:5]}. "
        f"Run 00_setup.ipynb or ensure the BioCause corpus is staged."
    )

# Step 1: Load the PubMedBERT tokenizer with all four entity markers BEFORE reconstructing candidates
pubmedbert_tokenizer = dl_utils.prepare_tokenizer_and_inputs(
    checkpoint_name=dl_utils.PUBMEDBERT_CHECKPOINT,
    max_length=dl_utils.DEFAULT_BIOCAUSE_MAX_LENGTH,
    additional_special_tokens=dl_utils.ENTITY_MARKERS,
)

# Verify all four entity markers are registered special tokens
for marker in dl_utils.ENTITY_MARKERS:
    tok_id = pubmedbert_tokenizer.convert_tokens_to_ids(marker)
    assert tok_id != pubmedbert_tokenizer.unk_token_id, f"Entity marker {marker} not recognized!"

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/28.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/226k [00:00<?, ?B/s]

In [12]:
# Verify marker insertion, span validation, and token budget on sample candidates
sample_rows = df_bio.head(5)

for idx, row in sample_rows.iterrows():
    sec_text = sec_text_cache[row["section_id"]]
    marked = dl_utils.reconstruct_biocause_context_with_markers(
        section_text=sec_text,
        arg1_start=int(row["arg1_start"]),
        arg1_end=int(row["arg1_end"]),
        arg2_start=int(row["arg2_start"]),
        arg2_end=int(row["arg2_end"]),
        arg1_text=str(row["arg1_text"]),
        arg2_text=str(row["arg2_text"]),
        candidate_id=str(row["candidate_id"]),
        max_length=dl_utils.DEFAULT_BIOCAUSE_MAX_LENGTH,
        tokenizer=pubmedbert_tokenizer,
    )
    assert "[E1]" in marked and "[/E1]" in marked, "Missing E1 markers"
    assert "[E2]" in marked and "[/E2]" in marked, "Missing E2 markers"
    tok_count = len(pubmedbert_tokenizer.encode(marked, add_special_tokens=True))
    assert tok_count <= dl_utils.DEFAULT_BIOCAUSE_MAX_LENGTH, f"Token count {tok_count} > 384"

In [13]:
# Reconstruct marked source context for all BioCause candidates using PubMedBERT tokenizer.
# Failed reconstructions (e.g. overlapping spans, out-of-bounds offsets, budget overflow) are
# flagged with reconstruction_valid = False and excluded from model-eligible splits.
reconstruction_valid = []
marked_texts = []
diagnostics = []

for idx, row in df_bio.iterrows():
    sec_text = sec_text_cache[row["section_id"]]
    try:
        marked = dl_utils.reconstruct_biocause_context_with_markers(
            section_text=sec_text,
            arg1_start=int(row["arg1_start"]),
            arg1_end=int(row["arg1_end"]),
            arg2_start=int(row["arg2_start"]),
            arg2_end=int(row["arg2_end"]),
            arg1_text=str(row["arg1_text"]),
            arg2_text=str(row["arg2_text"]),
            candidate_id=str(row["candidate_id"]),
            max_length=dl_utils.DEFAULT_BIOCAUSE_MAX_LENGTH,
            tokenizer=pubmedbert_tokenizer,
        )
        enc_len = len(pubmedbert_tokenizer.encode(marked, add_special_tokens=True))
        if enc_len > dl_utils.DEFAULT_BIOCAUSE_MAX_LENGTH:
            raise ValueError(
                f"Candidate {row['candidate_id']}: Encoded token length {enc_len} exceeds budget {dl_utils.DEFAULT_BIOCAUSE_MAX_LENGTH}"
            )
        reconstruction_valid.append(True)
        marked_texts.append(marked)
        diagnostics.append("")
    except Exception as e:
        reconstruction_valid.append(False)
        marked_texts.append(None)
        diagnostics.append(str(e))

df_bio["reconstruction_valid"] = reconstruction_valid
df_bio["marked_text"] = marked_texts
df_bio["diagnostic"] = diagnostics

df_invalid = df_bio[~df_bio["reconstruction_valid"]]
error_categories = []
for d in df_invalid["diagnostic"]:
    if "Overlapping" in d:
        error_categories.append("Overlapping argument spans")
    elif "Cropping failed" in d or "exceeds budget" in d or "exceeds max_tokens" in d:
        error_categories.append("Cropping failed to preserve spans within token budget")
    elif "Invalid arg" in d and "offset" in d:
        error_categories.append("Offset out of section bounds in standoff")
    elif "mismatch" in d:
        error_categories.append("Text span mismatch with source")
    else:
        error_categories.append("Other offset/span error")

diagnostics_summary_df = (
    pd.Series(error_categories, dtype="object")
    .value_counts()
    .rename_axis("Failure_Reason")
    .reset_index(name="Excluded_Candidates")
)

# Explicitly filter model-eligible candidates (strictly excluding invalid reconstructions)
df_bio_eligible = df_bio[df_bio["reconstruction_valid"]].copy()

# Verify model-eligible invariants and authoritative counts
assert (df_bio_eligible["reconstruction_valid"] == True).all(), "Non-valid reconstruction in model-eligible candidates!"
assert df_bio_eligible["marked_text"].notna().all(), "Null marked_text found in model-eligible candidates!"
assert df_bio_eligible["marked_text"].str.contains(r"\[E1\]").all(), "Missing [E1] marker!"
assert df_bio_eligible["marked_text"].str.contains(r"\[/E1\]").all(), "Missing [/E1] marker!"
assert df_bio_eligible["marked_text"].str.contains(r"\[E2\]").all(), "Missing [E2] marker!"
assert df_bio_eligible["marked_text"].str.contains(r"\[/E2\]").all(), "Missing [/E2] marker!"
assert df_bio_eligible["candidate_id"].duplicated().sum() == 0, "Duplicate candidate_ids in model-eligible set!"

for s, exp_counts in dl_utils.BIOCAUSE_MODEL_ELIGIBLE_COUNTS.items():
    sub_elig = df_bio_eligible[df_bio_eligible["split"] == s]
    assert len(sub_elig) == exp_counts["total"], (
        f"{s} model-eligible total: {len(sub_elig)} vs expected {exp_counts['total']}"
    )
    assert int((sub_elig["label"] == 1).sum()) == exp_counts["positive"], (
        f"{s} model-eligible positive mismatch"
    )
    assert int((sub_elig["label"] == 0).sum()) == exp_counts["negative"], (
        f"{s} model-eligible negative mismatch"
    )

max_tok_count = max(len(pubmedbert_tokenizer.encode(t, add_special_tokens=True)) for t in df_bio_eligible["marked_text"])
assert max_tok_count <= dl_utils.DEFAULT_BIOCAUSE_MAX_LENGTH, f"Max token count {max_tok_count} > 384!"

train_arts = set(df_bio_eligible[df_bio_eligible["split"] == "train"]["article_id"].unique())
val_arts = set(df_bio_eligible[df_bio_eligible["split"] == "validation"]["article_id"].unique())
test_arts = set(df_bio_eligible[df_bio_eligible["split"] == "test"]["article_id"].unique())
assert len(train_arts & val_arts) == 0, "Train/Val article leakage in eligible set!"
assert len(train_arts & test_arts) == 0, "Train/Test article leakage in eligible set!"
assert len(val_arts & test_arts) == 0, "Val/Test article leakage in eligible set!"

diagnostics_summary_df

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (919 > 384). Running this sequence through the model will result in indexing errors


,Failure_Reason,Excluded_Candidates
0,Overlapping argument spans,188
1,Offset out of section bounds in standoff,12
2,Cropping failed to preserve spans within token...,2


In [14]:
# Verify both argument orderings are represented in model-eligible candidates
offset_cols = ["arg1_start", "arg1_end", "arg2_start", "arg2_end"]
missing_offsets = [c for c in offset_cols if c not in df_bio_eligible.columns]
if missing_offsets:
    raise ValueError(
        f"BioCause candidates missing required offset columns: {missing_offsets}. "
        f"Offsets are strictly required for the entity marker workflow."
    )

arg2_first = df_bio_eligible[df_bio_eligible["arg2_start"] < df_bio_eligible["arg1_start"]]
arg1_first = df_bio_eligible[df_bio_eligible["arg1_start"] < df_bio_eligible["arg2_start"]]
assert len(arg1_first) > 0 and len(arg2_first) > 0, "Expected both argument orderings in BioCause"
assert len(arg1_first) + len(arg2_first) == len(df_bio_eligible), "Unexpected equal start offsets"

---

## Section E — Tokenizer Profiling
Load tokenizers, register entity markers as special tokens, profile token lengths,
calculate truncation statistics, and apply context cropping where necessary.

In [ ]:
# Section E: Tokenizer Profiling (DistilBERT & PubMedBERT)
# Profile sequence lengths across both datasets to document sequence length limits.

try:
    from transformers import AutoTokenizer
    HAS_TRANSFORMERS = True
except ImportError:
    HAS_TRANSFORMERS = False
    warnings.warn("transformers not installed; skipping tokenizer profiling.")

In [16]:
if HAS_TRANSFORMERS:
    # 1. Profile DistilBERT on Corr2Cause inputs
    distilbert_tokenizer = dl_utils.prepare_tokenizer_and_inputs(
        checkpoint_name=dl_utils.DISTILBERT_CHECKPOINT,
        max_length=dl_utils.DEFAULT_CORR2CAUSE_MAX_LENGTH,
    )

    sample_c2c_texts = [
        f"[CLS] {row['input']} [SEP]"
        for _, row in df_c2c_train.head(1000).iterrows()
    ]

    c2c_profile = dl_utils.profile_token_lengths(
        texts=sample_c2c_texts,
        tokenizer=distilbert_tokenizer,
        max_samples=1000,
        target_budget=dl_utils.DEFAULT_CORR2CAUSE_MAX_LENGTH,
    )

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (702 > 256). Running this sequence through the model will result in indexing errors


In [17]:
token_profile_df = None
if HAS_TRANSFORMERS:
    # 2. Profile PubMedBERT on marked BioCause inputs (model-eligible candidates)
    bio_profile = dl_utils.profile_token_lengths(
        texts=df_bio_eligible["marked_text"].head(1000).tolist(),
        tokenizer=pubmedbert_tokenizer,
        max_samples=1000,
        target_budget=dl_utils.DEFAULT_BIOCAUSE_MAX_LENGTH,
    )
    token_profile_df = pd.DataFrame([
        {
            "Dataset": "Corr2Cause (Train sample)",
            "Tokenizer": dl_utils.DISTILBERT_CHECKPOINT,
            "Samples": c2c_profile["count"],
            "Min": c2c_profile["min"],
            "Mean": c2c_profile["mean"],
            "Median": int(c2c_profile["median"]),
            "P95": int(c2c_profile["p95"]),
            "P99": int(c2c_profile["p99"]),
            "Max": c2c_profile["max"],
            "Budget": c2c_profile["target_budget"],
            "Within_Budget_%": c2c_profile["pct_within_budget"],
        },
        {
            "Dataset": "BioCause (Model-eligible sample)",
            "Tokenizer": dl_utils.PUBMEDBERT_CHECKPOINT,
            "Samples": bio_profile["count"],
            "Min": bio_profile["min"],
            "Mean": bio_profile["mean"],
            "Median": int(bio_profile["median"]),
            "P95": int(bio_profile["p95"]),
            "P99": int(bio_profile["p99"]),
            "Max": bio_profile["max"],
            "Budget": bio_profile["target_budget"],
            "Within_Budget_%": bio_profile["pct_within_budget"],
        },
    ])
token_profile_df

,Dataset,Tokenizer,Samples,Min,Mean,Median,P95,P99,Max,Budget,Within_Budget_%
0,Corr2Cause (Train sample),distilbert-base-uncased,1000,86,206.92,172,385,705,710,256,84.8
1,BioCause (Model-eligible sample),microsoft/BiomedNLP-BiomedBERT-base-uncased-ab...,1000,173,351.72,382,384,384,384,384,100.0


---

## Section F — Class Imbalance
Calculate the BioCause positive class weight using the training split only.

In [18]:
# Section F: Dynamic Training Class Weight Calculation & Data Preparation Summary
# Calculated strictly from model-eligible training candidates

df_bio_train_eligible = df_bio_eligible[df_bio_eligible['split'] == 'train']
n_pos_train = int((df_bio_train_eligible['label'] == 1).sum())
n_neg_train = int((df_bio_train_eligible['label'] == 0).sum())

pos_weight = dl_utils.calculate_dynamic_class_weight(
    n_negative=n_neg_train,
    n_positive=n_pos_train,
)
assert np.isclose(pos_weight, 9307 / 621), f'Unexpected pos_weight: {pos_weight}'

summary_rows = []
for split_name, df_split in [
    ('train', df_c2c_train),
    ('validation', df_c2c_val),
    ('test', df_c2c_test),
]:
    summary_rows.append({
        'Dataset': 'Corr2Cause',
        'Split': split_name,
        'Articles': '-',
        'Raw_Candidates': len(df_split),
        'Invalid_Excluded': 0,
        'Model_Eligible': len(df_split),
        'Positive': int((df_split['label'] == 1).sum()),
        'Negative': int((df_split['label'] == 0).sum()),
        'Pos_Class_Weight': '-',
    })

for split_name in ['train', 'validation', 'test']:
    sub_raw = df_bio[df_bio['split'] == split_name]
    sub_elig = df_bio_eligible[df_bio_eligible['split'] == split_name]
    summary_rows.append({
        'Dataset': 'BioCause',
        'Split': split_name,
        'Articles': int(sub_elig['article_id'].nunique()),
        'Raw_Candidates': len(sub_raw),
        'Invalid_Excluded': len(sub_raw) - len(sub_elig),
        'Model_Eligible': len(sub_elig),
        'Positive': int((sub_elig['label'] == 1).sum()),
        'Negative': int((sub_elig['label'] == 0).sum()),
        'Pos_Class_Weight': f'{pos_weight:.4f}' if split_name == 'train' else '-',
    })

summary_df = pd.DataFrame(summary_rows)
summary_df

,Dataset,Split,Articles,Raw_Candidates,Invalid_Excluded,Model_Eligible,Positive,Negative,Pos_Class_Weight
0,Corr2Cause,train,-,205734,0,205734,38048,167686,-
1,Corr2Cause,validation,-,1076,0,1076,139,937,-
2,Corr2Cause,test,-,1162,0,1162,180,982,-
3,BioCause,train,12,10086,158,9928,621,9307,14.9871
4,BioCause,validation,4,1914,30,1884,131,1753,-
5,BioCause,test,4,1408,14,1394,100,1294,-


---

## Section G — Export Processed Datasets
Export six model-ready CSV files to `data/processed/`.

In [19]:
# Section G: Export Model-Ready Datasets

PROCESSED_DIR = ROOT_DIR / 'data' / 'processed'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# --- Corr2Cause exports ---
df_c2c_train_out = df_c2c_train.copy()
df_c2c_train_out['split'] = 'train'
df_c2c_val_out = df_c2c_val.copy()
df_c2c_val_out['split'] = 'validation'
df_c2c_test_out = df_c2c_test.copy()
df_c2c_test_out['split'] = 'test'

c2c_exports = {
    'dl_corr2cause_train.csv': df_c2c_train_out,
    'dl_corr2cause_val.csv': df_c2c_val_out,
    'dl_corr2cause_test.csv': df_c2c_test_out,
}

for fname, df in c2c_exports.items():
    dl_utils.export_predictions(df, str(PROCESSED_DIR / fname), format_type='csv')

In [20]:
# --- BioCause exports ---
# Select columns needed by downstream DL notebooks (strictly model-eligible source-verified candidates)
biocause_export_cols = [
    col for col in [
        'candidate_id', 'article_id', 'section_id',
        'arg1_id', 'arg2_id', 'arg1_text', 'arg2_text',
        'arg1_start', 'arg1_end', 'arg2_start', 'arg2_end',
        'between_text', 'section_text', 'marked_text',
        'label', 'split', 'arg1_before_arg2',
    ] if col in df_bio_eligible.columns
]

bio_exports = {
    'dl_biocause_train.csv': df_bio_eligible[df_bio_eligible['split'] == 'train'][biocause_export_cols],
    'dl_biocause_val.csv': df_bio_eligible[df_bio_eligible['split'] == 'validation'][biocause_export_cols],
    'dl_biocause_test.csv': df_bio_eligible[df_bio_eligible['split'] == 'test'][biocause_export_cols],
}

for fname, df in bio_exports.items():
    dl_utils.export_predictions(df, str(PROCESSED_DIR / fname), format_type='csv')

# --- Export BioCause Reconstruction Diagnostics ---
# Separate diagnostics CSV containing all invalid candidates with reconstruction failure reasons
diag_cols = [
    col for col in [
        'candidate_id', 'article_id', 'section_id',
        'arg1_id', 'arg2_id', 'arg1_text', 'arg2_text',
        'arg1_start', 'arg1_end', 'arg2_start', 'arg2_end',
        'label', 'split', 'diagnostic',
    ] if col in df_bio.columns
]
df_diagnostics_export = df_bio[~df_bio['reconstruction_valid']][diag_cols]
dl_utils.export_predictions(
    df_diagnostics_export,
    str(PROCESSED_DIR / 'dl_biocause_reconstruction_diagnostics.csv'),
    format_type='csv',
)

In [21]:
# Final verification: read back all 6 model-ready exports and diagnostics file, verify schemas, counts, and token budget
expected_exports = {
    "dl_corr2cause_train.csv": (dl_utils.CORR2CAUSE_SPLIT_COUNTS["train"], ["input", "label", "split"]),
    "dl_corr2cause_val.csv": (dl_utils.CORR2CAUSE_SPLIT_COUNTS["validation"], ["input", "label", "split"]),
    "dl_corr2cause_test.csv": (dl_utils.CORR2CAUSE_SPLIT_COUNTS["test"], ["input", "label", "split"]),
    "dl_biocause_train.csv": (dl_utils.BIOCAUSE_MODEL_ELIGIBLE_COUNTS["train"]["total"], ["candidate_id", "marked_text", "label", "split"]),
    "dl_biocause_val.csv": (dl_utils.BIOCAUSE_MODEL_ELIGIBLE_COUNTS["validation"]["total"], ["candidate_id", "marked_text", "label", "split"]),
    "dl_biocause_test.csv": (dl_utils.BIOCAUSE_MODEL_ELIGIBLE_COUNTS["test"]["total"], ["candidate_id", "marked_text", "label", "split"]),
}

for fname, (exp_rows, req_cols) in expected_exports.items():
    fpath = PROCESSED_DIR / fname
    assert fpath.exists(), f"Missing required dataset export: {fname}"
    df_readback = pd.read_csv(fpath)
    actual_rows = len(df_readback)
    assert actual_rows == exp_rows, f"{fname} row mismatch: {actual_rows} vs expected {exp_rows}"
    dl_utils.validate_dataset_schema(df_readback, req_cols, dataset_name=fname)
    if "marked_text" in df_readback.columns:
        assert df_readback["marked_text"].notna().all(), f"{fname} contains null marked_text!"
        max_enc = max(len(pubmedbert_tokenizer.encode(t, add_special_tokens=True)) for t in df_readback["marked_text"])
        assert max_enc <= dl_utils.DEFAULT_BIOCAUSE_MAX_LENGTH, f"{fname} contains sequence exceeding 384 tokens ({max_enc})"

diag_fpath = PROCESSED_DIR / "dl_biocause_reconstruction_diagnostics.csv"
assert diag_fpath.exists(), "Missing reconstruction diagnostics export: dl_biocause_reconstruction_diagnostics.csv"
df_diag_readback = pd.read_csv(diag_fpath)
assert len(df_diag_readback) == len(df_diagnostics_export), (
    f"Diagnostics row mismatch: {len(df_diag_readback)} vs expected {len(df_diagnostics_export)}"
)
assert len(df_bio_eligible) + len(df_diag_readback) == len(df_bio), (
    "Model-eligible plus excluded diagnostics must equal raw BioCause candidate count"
)

## 5. Expected Outputs & Evaluation Criteria

- `data/processed/dl_corr2cause_{train,val,test}.csv`: Cleaned NLI splits conforming to canonical split counts (205,734 train; 1,076 validation; 1,162 test).
- `data/processed/dl_biocause_{train,val,test}.csv`: Source-verified, marked relation candidate datasets with preserved 12/4/4 article grouping (9,928 train; 1,884 validation; 1,394 test model-eligible candidates).
- `data/processed/dl_biocause_reconstruction_diagnostics.csv`: Audit log of the 202 raw BioCause candidates excluded due to invalid source reconstruction.
- **Evaluation Criteria**:
  - 100% split integrity: Zero overlap between article IDs in train, val, and test.
  - Argument marker completeness: Both entity spans intact post-formatting and tokenization within the 384-token budget.
  - Class weight verification: Dynamic training ratio $w_{\text{pos}} = 9,307 / 621 = 14.9871$ computed strictly from model-eligible training data.
